In [1]:
import joblib
import pandas as pd

df_users = pd.read_parquet("data\\11092026\\users.parquet")
df_movies = pd.read_parquet("data\\11092026\\movies.parquet")
df_interactions = pd.read_parquet("data\\11092026\\interactions.parquet")

encoders = joblib.load("data\\11092026\\encoders.pkl")
metadata = joblib.load("data\\11092026\\metadata.pkl")

In [2]:
df_users.head(10)

,user_id,gender,age_group,occupation,zip_code
0,1,0,1,10,48067
1,2,1,56,16,70072
2,3,1,25,15,55117
3,4,1,45,7,02460
4,5,1,25,20,55455
5,6,0,50,9,55117
6,7,1,35,1,06810
7,8,1,25,12,11413
8,9,1,25,17,61614
9,10,0,35,1,95370


In [3]:


df_movies.head(5)

,movie_id,title,genre,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,...,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),Animation|Children's|Comedy,0,0,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),Adventure|Children's|Fantasy,0,1,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),Comedy|Romance,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),Comedy|Drama,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,5,Father of the Bride Part II (1995),Comedy,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
df_interactions.head(10)

df_interactions["movie_id"] = (
    encoders["movie_encoder"].inverse_transform(df_interactions["movie_idx"])
)

df_interactions["user_id"] = (
    encoders["user_encoder"].inverse_transform(df_interactions["user_idx"])
)

df_interactions


,user_idx,movie_idx,rating,timestamp,movie_id,user_id
0,0,1104,5.0,978300760,1193,1
1,0,639,3.0,978302109,661,1
2,0,853,3.0,978301968,914,1
3,0,3177,4.0,978300275,3408,1
4,0,2162,5.0,978824291,2355,1
...,...,...,...,...,...,...
1000204,6039,1019,1.0,956716541,1091,6040
1000205,6039,1022,5.0,956704887,1094,6040
1000206,6039,548,5.0,956704746,562,6040
1000207,6039,1024,4.0,956715648,1096,6040


In [ ]:
metadata


{'n_movies': 3706, 'n_users': 6040}

In [ ]:
encoders


{'user_encoder': LabelEncoder(), 'movie_encoder': LabelEncoder()}

In [ ]:
df_full.columns


NameError: name 'df_full' is not defined

In [8]:
df_full = df_users.merge(df_interactions, on="user_id", how="left")
df_full = df_full.merge(df_movies, on="movie_id", how="left")



In [60]:
df_full

,user_id,gender,age_group,occupation,zip_code,user_idx,movie_idx,rating,timestamp,movie_id,...,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year,title_clean,rating_binary
0,1,0,1,10,48067,0,1104,5.0,978300760,1193,...,0,0,0,0,0,0,0,1975,One Flew Over the Cuckoo's Nest,1
1,1,0,1,10,48067,0,639,3.0,978302109,661,...,1,0,0,0,0,0,0,1996,James and the Giant Peach,0
2,1,0,1,10,48067,0,853,3.0,978301968,914,...,1,0,1,0,0,0,0,1964,My Fair Lady,0
3,1,0,1,10,48067,0,3177,4.0,978300275,3408,...,0,0,0,0,0,0,0,2000,Erin Brockovich,1
4,1,0,1,10,48067,0,2162,5.0,978824291,2355,...,0,0,0,0,0,0,0,1998,"Bug's Life, A",1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000204,6040,1,25,6,11106,6039,1019,1.0,956716541,1091,...,0,0,0,0,0,0,0,1989,Weekend at Bernie's,0
1000205,6040,1,25,6,11106,6039,1022,5.0,956704887,1094,...,0,0,1,0,0,1,0,1992,"Crying Game, The",1
1000206,6040,1,25,6,11106,6039,548,5.0,956704746,562,...,0,0,0,0,0,0,0,1995,Welcome to the Dollhouse,1
1000207,6040,1,25,6,11106,6039,1024,4.0,956715648,1096,...,0,0,0,0,0,0,0,1982,Sophie's Choice,1


In [ ]:
df_full.columns

drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "rating",
    "rating_binary"
]
df_full.drop(columns=drop_cols)

In [9]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.temporal_split import TemporalSplit


In [123]:
def temporal_stats(train):

    df = train.copy().sort_values(["movie_id", "timestamp"])
    df["_sq_rating"] = df["rating"]**2

    g = df.groupby("movie_id", sort=False)

    movie_count = g.cumcount()
    previous_sum_rating = g["rating"].cumsum() - df["rating"]
    previous_sq_sum_rating = g["_sq_rating"].cumsum() - df["_sq_rating"]

    df["count_movie_rating"] = movie_count
    df["avg_movie_rating"] = previous_sum_rating / movie_count
    # df["std_movie_rating"] = np.sqrt(
    #     (previous_sq_sum_rating - (previous_sum_rating)**2 / movie_count)
    #     / (movie_count - 1)
    # ).where(movie_count > 1)

    return df.sort_values(by="user_id")

In [124]:
import numpy as np

df_full["rating_binary"] = (df_full["rating"] >= 4).astype(int)
df_full["year"] = (df_full["title"].str.extract(r"\((\d+)\)")).astype(int)

drop_cols = [
    "movie_idx",
    "user_idx",
    "genre",
    "title",
    "timestamp",
    "zip_code",
    "rating",
    "rating_binary",
    "title"
]

cat_features = [
    "user_id",
    "movie_id",
    "gender",
    "age_group",
    "occupation",
]

train , val , test = TemporalSplit().split(data=df_full)

train = temporal_stats(train)

group_train = train["user_id"]
group_val = val["user_id"]

y_train = train["rating_binary"]
y_val = val["rating_binary"]
y_test = test["rating_binary"]

X_train = train.drop(columns=drop_cols)
X_val = val.drop(columns=drop_cols)
X_test = test.drop(columns=drop_cols)

X_train

,user_id,gender,age_group,occupation,movie_id,Action,Adventure,Animation,Children's,Comedy,...,Mystery,Romance,Sci-Fi,Thriller,War,Western,year,_sq_rating,count_movie_rating,avg_movie_rating
0,1,0,1,10,3186,0,0,0,0,0,...,0,0,0,0,0,0,1999,16.0,309,3.449838
2,1,0,1,10,1721,0,0,0,0,0,...,0,1,0,0,0,0,1997,16.0,1255,3.571315
9,1,0,1,10,1193,0,0,0,0,0,...,0,0,0,0,0,0,1975,25.0,1471,4.390211
7,1,0,1,10,2804,0,0,0,0,1,...,0,0,0,0,0,0,1983,25.0,1149,4.240209
13,1,0,1,10,608,0,0,0,0,0,...,0,0,0,1,0,0,1996,16.0,2134,4.261012
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
797671,6040,1,25,6,2147,0,0,0,0,0,...,0,0,0,0,0,0,1986,4.0,1,2.000000
797606,6040,1,25,6,2150,0,0,0,0,1,...,0,0,0,0,0,0,1980,9.0,0,NaN
797548,6040,1,25,6,290,0,0,0,0,0,...,0,0,0,0,0,0,1994,16.0,0,NaN
797730,6040,1,25,6,1,0,0,1,1,1,...,0,0,0,0,0,0,1995,9.0,40,4.050000


In [105]:
from catboost import CatBoostRanker
import joblib

def fit(model_name="test"):

    model = CatBoostRanker(
        loss_function="YetiRank",
        eval_metric="NDCG:top=10",
        iterations=500,
        learning_rate=0.01,
        depth=5,
        random_seed=42,
        verbose=50
    )

    model.fit(
        X_train,
        y_train,
        group_id=group_train,
        cat_features=cat_features
    )

    joblib.dump(
        model,
        f"data\\models\\{model_name}.pkl"
    )

In [39]:
import joblib

model_s = {
    "model": model
}

joblib.dump(
    model_s,
    "data\\models\\baseline.pkl"
)

['data\\models\\baseline.pkl']

In [ ]:
import joblib

model = joblib.load("data\\models\\baseline.pkl")["model"]
model

CatBoostRanker(depth=5, eval_metric='NDCG:top=10', iterations=500, learning_rate=0.01, loss_function='YetiRank', random_seed=42, verbose=50)

In [ ]:
def movie_stats(history):
    table_movie = (
        history
        .groupby("movie_id")
        .agg(
            avg_movie_rating=("rating", "mean"),
            count_movie_rating=("rating", "count"),
            #std_movie_rating=("rating", "std"),
        )
        .reset_index()
    )

    # table_user = (
    #         history
    #         .groupby("user_id")
    #         .agg(
    #             avg_user_rating=("rating", "mean"),
    #             count_user_rating=("rating", "count"),
    #         )
    #         .reset_index()
    #     )

    return table_movie#, table_user


In [ ]:
ROOT_DIR = Path.cwd().parent
sys.path.append(str(ROOT_DIR))

from src.metrics.metrics import (
    recall_at_k,
    precision_at_k,
    ndcg_at_k,
)
import numpy as np

drop_cols = [
    "genre",
    "zip_code",
    "title",
]

def eval(
        model,
        train,
        val,
        df_movie,
        df_users,
        df_interactions,
        k=10,
        model_name="model"
):

    movies_ids = df_movie.movie_id.unique()

    train_seen = (
        train
        .groupby("user_id")["movie_id"]
        .agg(set)
        .to_dict()
    )

    recalls = []
    precisions = []
    ndcgs = []

    train_movie_stats = movie_stats(train) #, train_user_stats

    movie_features = df_movie.copy()

    movie_features["year"] = (
        movie_features["title"]
        .str.extract(r"\((\d{4})\)")
        .astype("Int64")
    )

    movie_features = movie_features.merge(train_movie_stats, on="movie_id", how="left")
    movie_features = movie_features.drop(columns=["genre", "title"])

    user_features = df_users.copy()
    #user_features = user_features.merge(train_user_stats, on="user_id", how='left')
    user_features = user_features.drop(columns="zip_code")

    for user_id in val.user_id.unique():

        watched = train_seen.get(user_id, set())

        candidates = movie_features[
            ~movie_features["movie_id"].isin(watched)
        ]

        candidates["user_id"] = user_id

        candidates = candidates.merge(user_features, on="user_id", how="left")

        candidates = candidates[X_train.columns]

        scores = model.predict(candidates)

        top_indices = np.argsort(scores)[::-1][:k]

        recommended = (
            candidates["movie_id"]
            .iloc[top_indices]
            .tolist()
        )

        val_per_user = val[
            val["user_id"] == user_id
        ]

        relevant = val_per_user.loc[
            val_per_user["rating"] >= 4,
            "movie_id"
        ]

        recall = recall_at_k(relevant, recommended)
        precision = precision_at_k(relevant, recommended)
        ndcg = ndcg_at_k(relevant, recommended)

        recalls.append(recall)
        precisions.append(precision)
        ndcgs.append(ndcg)

    metrics = {
        "recall": np.mean(recalls),
        "precision": np.mean(precisions),
        "ndcg": np.mean(ndcgs)
    }

    joblib.dump(
        metrics,
        f"data\\metrics\\{model_name}.pkl"
    )

    return metrics


In [ ]:

fit("temporal_split_1")

Groupwise loss function. OneHotMaxSize set to 10
0:	total: 734ms	remaining: 6m 6s
50:	total: 36.8s	remaining: 5m 23s
100:	total: 1m 10s	remaining: 4m 36s
150:	total: 1m 42s	remaining: 3m 56s


In [43]:
res = eval(model, train, val, df_movies, df_users, df_interactions, model_name="baseline_metrics")

In [44]:
res

{'recall': np.float64(0.023557995863224122),
 'precision': np.float64(0.01867549668874172),
 'ndcg': np.float64(0.02542515150375308)}

In [74]:
df = pd.DataFrame({
    "movie_id": [10, 10, 10, 20, 20],
    "timestamp": [1, 2, 3, 1, 2],
    "rating": [5, 3, 4, 2, 5]
})
df

,movie_id,timestamp,rating
0,10,1,5
1,10,2,3
2,10,3,4
3,20,1,2
4,20,2,5


In [118]:
user_id = 123
movie_id = [10, 11]

df = train[train["movie_id"].isin(movie_id)].sort_values(by="timestamp").reset_index(drop=True)
df["_sq_rating"] = df["rating"]**2

g = df.groupby("movie_id", sort=False)
movie_count = g.cumcount()
previous_sum_rating = g["rating"].cumsum() - df["rating"]
previous_sq_sum_rating = g["_sq_rating"].cumsum() - df["_sq_rating"]

df["count_movie_rating"] = movie_count
df["avg_movie_rating"] = previous_sum_rating / movie_count
df["std_movie_rating"] = np.sqrt(
    (previous_sq_sum_rating - (previous_sum_rating)**2 / movie_count)
    / (movie_count - 1)
).where(movie_count > 1)

print(df[["movie_id", "count_movie_rating", "avg_movie_rating", "std_movie_rating"]])
df

      movie_id  count_movie_rating  avg_movie_rating  std_movie_rating
0           11                   0               NaN               NaN
1           10                   0               NaN               NaN
2           11                   1          4.000000               NaN
3           11                   2          4.000000          0.000000
4           10                   1          5.000000               NaN
...        ...                 ...               ...               ...
1596        11                 851          3.795535          0.868009
1597        11                 852          3.796948          0.868479
1598        10                 745          3.530201          0.890187
1599        10                 746          3.529491          0.889801
1600        11                 853          3.797186          0.867997

[1601 rows x 4 columns]


,user_id,gender,age_group,occupation,zip_code,user_idx,movie_idx,rating,timestamp,movie_id,...,Sci-Fi,Thriller,War,Western,rating_binary,year,_sq_rating,count_movie_rating,avg_movie_rating,std_movie_rating
0,6035,0,25,1,78734,6034,10,4.0,956712108,11,...,0,0,0,0,1,1995,16.0,0,NaN,NaN
1,6027,1,18,4,20742,6026,9,5.0,956726620,10,...,0,1,0,0,1,1995,25.0,0,NaN,NaN
2,6025,0,25,1,32607,6024,10,4.0,956731215,11,...,0,0,0,0,1,1995,16.0,1,4.000000,NaN
3,6036,0,25,15,32603,6035,10,3.0,956752783,11,...,0,0,0,0,0,1995,9.0,2,4.000000,0.000000
4,6019,1,25,0,10024,6018,9,3.0,956761104,10,...,0,1,0,0,0,1995,9.0,1,5.000000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1596,1755,0,18,4,77005,1754,10,5.0,1030043772,11,...,0,0,0,0,1,1995,25.0,851,3.795535,0.868009
1597,1194,0,25,1,95617,1193,10,4.0,1036260379,11,...,0,0,0,0,1,1995,16.0,852,3.796948,0.868479
1598,184,0,25,0,19001,183,9,3.0,1038969828,10,...,0,1,0,0,0,1995,9.0,745,3.530201,0.890187
1599,102,1,35,19,20871,101,9,2.0,1039275574,10,...,0,1,0,0,0,1995,4.0,746,3.529491,0.889801
